In [ ]:
import os
import h5py
import matplotlib.pyplot as plt
import numpy as np
import torch
from models.skip1d import*
from tqdm.notebook import tqdm
import time
from matplotlib.patches import Rectangle

device = 'cpu'
torch.set_default_dtype(torch.float32)

base_path_noisy = r"E:\\bma22_epm_motioncorrected_round1_cropped_correction.mat"

with h5py.File(base_path_noisy, 'r') as mat_data:
    print("Available keys in .mat file:", list(mat_data.keys()))

    
    dataset = mat_data['Y']
    data_array = dataset[()]
    data_subset = data_array[800:1800, :, :]

vol_data = data_subset.astype(np.float32)

# Normalize the full video globally
full_video = torch.tensor(vol_data, dtype=torch.float32)
mean_val = full_video.mean()
std_val = full_video.std()

full_video_norm = (full_video - mean_val) / std_val

# Subsample 200 frames
video_tensor = full_video_norm

# video_tensor = full_video_norm[:, ]

H, W, T = video_tensor.shape[1], video_tensor.shape[2], video_tensor.shape[0]

epochs = 1000

##### signal componenet #####

signal_rank_H = 32
signal_rank_W = 32
signal_rank_T = 32
learning_rate = 1e-4

signal_h_inp = get_1d_posencode_inp(H, signal_rank_H//2)
signal_w_inp = get_1d_posencode_inp(W, signal_rank_W//2)
signal_t_inp = get_1d_posencode_inp(T, signal_rank_T//2)

skip_n33d = 128  # Number of channels in the downsampling path
skip_n33u = 128  # Number of channels in the upsampling path
skip_n11 = 4   # Number of channels in the skip connection
num_scales = 5  # Number of scales in the skip connection
upsample_mode = 'linear'  # Upsampling mode
downsample_mode = 'stride'  # Downsampling mode
pad = 'reflection'  # Padding mode
act_fun = 'LeakyReLU'  # Activation function
filter_size_up=3
filter_size_down=3 
filter_size_skip=1

signal_h_net = skip1d(
        signal_rank_H,                       # Dimensionality of positional encoding (e.g., 32)
        signal_rank_H,                       # Output dimensionality = Tucker rank (e.g., 32)
        num_channels_down = [skip_n33d]*num_scales if isinstance(skip_n33d, int) else skip_n33d,
        num_channels_up   = [skip_n33u]*num_scales if isinstance(skip_n33u, int) else skip_n33u,
        num_channels_skip = [skip_n11]*num_scales if isinstance(skip_n11, int) else skip_n11,
        upsample_mode     = upsample_mode,
        downsample_mode   = downsample_mode,
        need_sigmoid      = False,
        need_bias         = True,
        pad               = pad,
        act_fun           = act_fun,
        filter_size_up    = filter_size_up,
        filter_size_down  = filter_size_down,
        filter_skip_size  = filter_size_skip
    )

signal_w_net = skip1d(
        signal_rank_W,                       # Dimensionality of positional encoding (e.g., 32)
        signal_rank_W,                       # Output dimensionality = Tucker rank (e.g., 32)
        num_channels_down = [skip_n33d]*num_scales if isinstance(skip_n33d, int) else skip_n33d,
        num_channels_up   = [skip_n33u]*num_scales if isinstance(skip_n33u, int) else skip_n33u,
        num_channels_skip = [skip_n11]*num_scales if isinstance(skip_n11, int) else skip_n11,
        upsample_mode     = upsample_mode,
        downsample_mode   = downsample_mode,
        need_sigmoid      = False,
        need_bias         = True,
        pad               = pad,
        act_fun           = act_fun,
        filter_size_up    = filter_size_up,
        filter_size_down  = filter_size_down,
        filter_skip_size  = filter_size_skip
    )

signal_t_net = skip1d(
    signal_rank_T,                       # Dimensionality of positional encoding (e.g., 32)
    signal_rank_T,                       # Output dimensionality = Tucker rank (e.g., 32)
    num_channels_down = [skip_n33d]*num_scales if isinstance(skip_n33d, int) else skip_n33d,
    num_channels_up   = [skip_n33u]*num_scales if isinstance(skip_n33u, int) else skip_n33u,
    num_channels_skip = [skip_n11]*num_scales if isinstance(skip_n11, int) else skip_n11,
    upsample_mode     = upsample_mode,
    downsample_mode   = downsample_mode,
    need_sigmoid      = False,
    need_bias         = True,
    pad               = pad,
    act_fun           = act_fun,
    filter_size_up    = filter_size_up,
    filter_size_down  = filter_size_down,
    filter_skip_size  = filter_size_skip
)

#### global background component ####

global_rank_H = 6
global_rank_W = 6
global_rank_T = 1

global_h_inp = get_1d_posencode_inp(H, global_rank_H//2)
global_w_inp = get_1d_posencode_inp(H, global_rank_H//2)
global_t_inp = Constant1D(T)

skip_n33d = 128  # Number of channels in the downsampling path
skip_n33u = 128  # Number of channels in the upsampling path
skip_n11 = 4   # Number of channels in the skip connection
num_scales = 5  # Number of scales in the skip connection
upsample_mode = 'linear'  # Upsampling mode
downsample_mode = 'stride'  # Downsampling mode
pad = 'reflection'  # Padding mode
act_fun = 'LeakyReLU'  # Activation function
filter_size_up=3
filter_size_down=3 
filter_size_skip=1


global_h_net = skip1d(
    global_rank_H,                       # Dimensionality of positional encoding (e.g., 32)
    global_rank_H,                       # Output dimensionality = Tucker rank (e.g., 32)
    num_channels_down = [skip_n33d]*num_scales if isinstance(skip_n33d, int) else skip_n33d,
    num_channels_up   = [skip_n33u]*num_scales if isinstance(skip_n33u, int) else skip_n33u,
    num_channels_skip = [skip_n11]*num_scales if isinstance(skip_n11, int) else skip_n11,
    upsample_mode     = upsample_mode,
    downsample_mode   = downsample_mode,
    need_sigmoid      = False,
    need_bias         = True,
    pad               = pad,
    act_fun           = act_fun,
    filter_size_up    = filter_size_up,
    filter_size_down  = filter_size_down,
    filter_skip_size  = filter_size_skip
)

global_w_net = skip1d(
    global_rank_W,                       # Dimensionality of positional encoding (e.g., 32)
    global_rank_W,                       # Output dimensionality = Tucker rank (e.g., 32)
    num_channels_down = [skip_n33d]*num_scales if isinstance(skip_n33d, int) else skip_n33d,
    num_channels_up   = [skip_n33u]*num_scales if isinstance(skip_n33u, int) else skip_n33u,
    num_channels_skip = [skip_n11]*num_scales if isinstance(skip_n11, int) else skip_n11,
    upsample_mode     = upsample_mode,
    downsample_mode   = downsample_mode,
    need_sigmoid      = False,
    need_bias         = True,
    pad               = pad,
    act_fun           = act_fun,
    filter_size_up    = filter_size_up,
    filter_size_down  = filter_size_down,
    filter_skip_size  = filter_size_skip
)

global_core = torch.nn.Parameter(torch.ones((global_rank_T, global_rank_H, global_rank_W), dtype=torch.float32) /
    (global_rank_T * global_rank_H * global_rank_W) ** 0.5)

#### local background component ####

local_rank_H = 4
local_rank_W = 4
local_rank_T = 16

local_h_inp = get_1d_posencode_inp(H, local_rank_H//2)
local_w_inp = get_1d_posencode_inp(W, local_rank_W//2)
local_t_inp = get_1d_posencode_inp(T, local_rank_T//2)

skip_n33d = 128  # Number of channels in the downsampling path
skip_n33u = 128  # Number of channels in the upsampling path
skip_n11 = 4   # Number of channels in the skip connection
num_scales = 5  # Number of scales in the skip connection
upsample_mode = 'linear'  # Upsampling mode
downsample_mode = 'stride'  # Downsampling mode
pad = 'reflection'  # Padding mode
act_fun = 'LeakyReLU'  # Activation function
filter_size_up=3
filter_size_down=3 
filter_size_skip=1


local_h_net = skip1d(
    local_rank_H,                       # Dimensionality of positional encoding (e.g., 32)
    local_rank_H,                       # Output dimensionality = Tucker rank (e.g., 32)
    num_channels_down = [skip_n33d]*num_scales if isinstance(skip_n33d, int) else skip_n33d,
    num_channels_up   = [skip_n33u]*num_scales if isinstance(skip_n33u, int) else skip_n33u,
    num_channels_skip = [skip_n11]*num_scales if isinstance(skip_n11, int) else skip_n11,
    upsample_mode     = upsample_mode,
    downsample_mode   = downsample_mode,
    need_sigmoid      = False,
    need_bias         = True,
    pad               = pad,
    act_fun           = act_fun,
    filter_size_up    = filter_size_up,
    filter_size_down  = filter_size_down,
    filter_skip_size  = filter_size_skip
)

local_w_net = skip1d(
    local_rank_W,                       # Dimensionality of positional encoding (e.g., 32)
    local_rank_W,                       # Output dimensionality = Tucker rank (e.g., 32)
    num_channels_down = [skip_n33d]*num_scales if isinstance(skip_n33d, int) else skip_n33d,
    num_channels_up   = [skip_n33u]*num_scales if isinstance(skip_n33u, int) else skip_n33u,
    num_channels_skip = [skip_n11]*num_scales if isinstance(skip_n11, int) else skip_n11,
    upsample_mode     = upsample_mode,
    downsample_mode   = downsample_mode,
    need_sigmoid      = False,
    need_bias         = True,
    pad               = pad,
    act_fun           = act_fun,
    filter_size_up    = filter_size_up,
    filter_size_down  = filter_size_down,
    filter_skip_size  = filter_size_skip
)

local_t_net = skip1d(
    local_rank_T,                       # Dimensionality of positional encoding (e.g., 32)
    local_rank_T,                       # Output dimensionality = Tucker rank (e.g., 32)
    num_channels_down = [skip_n33d]*num_scales if isinstance(skip_n33d, int) else skip_n33d,
    num_channels_up   = [skip_n33u]*num_scales if isinstance(skip_n33u, int) else skip_n33u,
    num_channels_skip = [skip_n11]*num_scales if isinstance(skip_n11, int) else skip_n11,
    upsample_mode     = upsample_mode,
    downsample_mode   = downsample_mode,
    need_sigmoid      = False,
    need_bias         = True,
    pad               = pad,
    act_fun           = act_fun,
    filter_size_up    = filter_size_up,
    filter_size_down  = filter_size_down,
    filter_skip_size  = filter_size_skip
)

local_core = torch.nn.Parameter(torch.ones((local_rank_T, local_rank_H, local_rank_W), dtype=torch.float32) /
    (local_rank_T * local_rank_H * local_rank_W) ** 0.5)

##### end signal, global, and local background #####

# initialize training mode
signal_h_net.train()
signal_w_net.train()
signal_t_net.train()
global_h_net.train()
global_w_net.train()
local_h_net.train()
local_w_net.train()
local_t_net.train()

all_params = (
    list(signal_h_net.parameters()) +
    list(signal_w_net.parameters()) +
    list(signal_t_net.parameters()) +
    [signal_h_inp] +
    [signal_w_inp] +
    [signal_t_inp] +
    list(global_h_net.parameters()) +
    list(global_w_net.parameters()) +
    [global_h_inp] +
    [global_w_inp] +
    list(global_t_inp.parameters()) +
    [global_core] +
    list(local_h_net.parameters()) +
    list(local_w_net.parameters()) +
    list(local_t_net.parameters()) +
    [local_core]
)

criterion_l1 = L2Norm()
    
loss_array = np.zeros(epochs)
mse_array = np.zeros(epochs)    
time_array = np.zeros(epochs)

optimizer = torch.optim.Adam(lr=learning_rate, params=all_params)
    
# Create a learning scheduler
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size= epochs)

best_loss = float('inf')
best_epoch = 0
tic = time.time()

tbar = tqdm(range(epochs))

# frame_idx = 100  # Example frame index to visualize
# x, y = 65, 50      # top-left corner (column, row)
# width, height = 50, 52

# def plot_frame_with_rectangle(frame, x, y, width, height):
#     fig, ax = plt.subplots(figsize=(6, 6))
#     ax.imshow(frame, cmap='hot')
    
#     # Add rectangle patch
#     rect = Rectangle((x, y), width, height, linewidth=2, edgecolor='b', facecolor='none')
#     ax.add_patch(rect)

#     ax.set_title(f"Rectangle at ({x}, {y}), w={width}, h={height}")
#     plt.grid(False)
#     plt.axis('off')
#     plt.show()

loss_array = np.zeros(epochs)

def compute_softmax_neuropil_weight_mask(X: torch.Tensor, A_t: torch.Tensor, beta: float = 10.0, tau: float = 0.25) -> torch.Tensor:
    """
    Computes a soft neuropil weight mask using softmax-weighted average of correlation scores.

    Args:
        X     : (P, T) tensor of pixel traces (P = #pixels, T = timepoints)
        A_t   : (R, T) tensor of signal temporal factors (R = rank of signal component)
        beta  : Softmax temperature (higher = sharper attention)
        tau   : Sigmoid threshold to binarize or weigh final score

    Returns:
        W     : (P,) tensor of soft weights in [0, 1] for each pixel
    """

    # Step 1: Zero-center
    X_centered   = X - X.mean(dim=1, keepdim=True)      # (P, T)
    A_t_centered = A_t - A_t.mean(dim=1, keepdim=True)  # (R, T)

    # Step 2: Normalize
    X_norm   = F.normalize(X_centered, dim=1)    # (P, T)
    A_t_norm = F.normalize(A_t_centered, dim=1)  # (R, T)

    # Step 3: Pearson correlation: (P, T) x (T, R) => (P, R)
    corr = torch.matmul(X_norm, A_t_norm.T)      # (P, R)

    # Optional: keep only positive correlations (comment out if full range needed)
    corr = torch.clamp(corr, min=0.0)            # (P, R)

    # Step 4: Softmax over signal components (R) for each pixel
    attn_weights = torch.softmax(beta * corr, dim=1)    # (P, R)

    # Step 5: Softmax-weighted average of correlations
    soft_score = (corr * attn_weights).sum(dim=1)       # (P,)

    # Step 6: Optional sigmoid thresholding (differentiable gating)
    W = torch.sigmoid(beta * (soft_score - tau))        # (P,) ∈ [0, 1]

    return W

for idx in tbar:
    A_h = h_net(h_inp).squeeze(0).permute(1, 0)  # [H, 32]
    A_w = w_net(w_inp).squeeze(0).permute(1, 0)  # [W, 32]
    A_t = t_net(t_inp).squeeze(0).permute(1, 0)  # [T, 32]

    # recon = torch.einsum('tr,hr,wr,rxy->thw', A_t, A_h, A_w, core)  # (T,H,W)

    # 1. Multiply along temporal mode (mode-0)
    out = torch.einsum('tr, rhw -> thw', A_t, core)      # [T, R_H, R_W]

    # 2. Multiply along spatial height (mode-1)
    out = torch.einsum('trw, hr -> t w h', out, A_h)     # [T, H, R_W]

    # 3. Multiply along spatial width (mode-2)
    recon = torch.einsum('trh, wr -> thw', out, A_w)     # [T, H, W]

    #### background spatial component smoothing loss ####
    lap = ((Zg[:,1:]-Zg[:,:-1])**2).sum() + ((Zg[1:]-Zg[:-1])**2).sum()

    loss = criterion_l1(video_tensor - recon)

    # Track metrics
    loss_array[idx] = loss.item()
    time_array[idx] = time.time() - tic

    if loss.item() < best_loss:
        best_loss = loss.item()
        best_epoch = idx
        best_reconstruction = recon.detach().clone()

    # Backprop
    optimizer.zero_grad()
    torch.autograd.set_detect_anomaly(True)
    loss.backward()
    optimizer.step()
    scheduler.step()  # optional

    loss_array[idx] = loss.item()

    # Optionally visualize or print
    if idx % 10 == 0:
        print(f'Epoch {idx:4d} | loss {loss.item():.4e}')
    
    if idx % 50 == 0 or idx == 0:
        recon_scaled = recon* std_val + mean_val
        video_scaled = video_tensor * std_val + mean_val
        recon_frame = recon_scaled[frame_idx].detach().cpu().numpy()
        gt_frame = video_scaled[frame_idx].detach().cpu().numpy()

        diff_frame = np.abs(gt_frame - recon_frame)
        diff_frame_signed = gt_frame - recon_frame

        fig, axs = plt.subplots(1, 4, figsize=(12, 4))
        axs[0].imshow(gt_frame, cmap='hot')
        axs[0].set_title('Noisy Input Frame')
        axs[0].axis('off')

        axs[1].imshow(recon_frame, cmap='hot')
        axs[1].set_title('Reconstructed Frame')
        axs[1].axis('off')

        axs[2].imshow(diff_frame, cmap='hot')
        axs[2].set_title('Abs Difference')
        axs[2].axis('off')

        axs[3].imshow(diff_frame_signed, cmap='hot')
        axs[3].set_title('Signed Difference')
        axs[3].axis('off')

        plt.suptitle(f'Epoch {idx}')
        plt.tight_layout()
        plt.show()

        plot_frame_with_rectangle(gt_frame, x, y, width, height)

        # Extract the patch over time (shape: T x height x width)
        patch_noisy = video_scaled[:, y:y+height, x:x+width]  # shape: (T, h, w)
        patch_recon = recon_scaled[:, y:y+height, x:x+width]

        # Average over spatial dimensions
        avg_trace_noisy = patch_noisy.mean(dim=(1, 2)).detach().cpu().numpy()
        avg_trace_recon = patch_recon.mean(dim=(1, 2)).detach().cpu().numpy()

        # Plot the time trace
        plt.figure(figsize=(10, 4))
        plt.plot(avg_trace_noisy, label='Noisy Mean Intensity in  ROI', alpha=0.6)
        plt.plot(avg_trace_recon, label='Reconstructed Mean Intensity in ROI', color='orange')
        plt.xlabel("Time (frames)")
        plt.ylabel("Average Intensity")
        plt.title(f"Time Trace for ROI at ({x}, {y}), w={width}, h={height}")
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plt.show()

#plot loss
plt.figure(figsize=(10, 4))
plt.plot(loss_array, label='Loss', color='blue')
plt.xlabel("Epoch")
plt.ylabel("Loss Value")
plt.title("Training Loss Over Epochs")